# 🔧 Stage 2 — Data Enrichment

**Goal:** Enrich parsed recipe data with additional derived attributes.

This notebook covers:
- **Unit normalisation** (e.g. `tablespoon → tbsp`)
- **Ingredient name normalisation** (e.g. `granulated sugar → sugar`)
- **SI unit conversion** (cups → ml, oz → g, lb → g)
- **Duplicate detection** via MD5 content fingerprinting
- **Auto-tagging** (quick, baking, vegan-friendly, protein-rich, …)
- Saving `enriched_data.json`

---


## 0. Setup

In [ ]:
import sys, json, os

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import pandas as pd
from src.enrichment import (
    convert_ingredient_to_si,
    convert_recipe_to_si,
    detect_duplicates,
    generate_tags,
    enrich,
)
from src.config import UNIT_MAP, INGREDIENT_NORM, SI_CONVERSION

INPUT_FILE  = os.path.join(REPO_ROOT, 'data', 'parsed_data.json')
OUTPUT_FILE = os.path.join(REPO_ROOT, 'data', 'enriched_data.json')

print('Setup complete.')


In [ ]:
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    recipes = json.load(f)
print(f'Loaded {len(recipes)} parsed recipes from {INPUT_FILE}')


## 1. Unit Normalisation Map

All unit aliases that will be standardised:

In [ ]:
pd.DataFrame(
    [{'alias': k, 'canonical': v} for k, v in UNIT_MAP.items()]
).sort_values('canonical').reset_index(drop=True)


## 2. Ingredient Name Normalisation Map

In [ ]:
pd.DataFrame(
    [{'alias': k, 'canonical': v} for k, v in INGREDIENT_NORM.items()]
).sort_values('canonical').reset_index(drop=True)


## 3. SI Unit Conversion Demo

In [ ]:
demo_ingredients = [
    {'quantity': 2.0,   'unit': 'cup',  'item': 'flour',         'original_text': '2 cups flour'},
    {'quantity': 1.5,   'unit': 'tbsp', 'item': 'olive oil',     'original_text': '1.5 tbsp olive oil'},
    {'quantity': 500.0, 'unit': 'g',    'item': 'potatoes',      'original_text': '500 g potatoes'},
    {'quantity': 1.0,   'unit': 'lb',   'item': 'beef',          'original_text': '1 lb beef'},
    {'quantity': 0.75,  'unit': 'cup',  'item': 'sugar',         'original_text': '3/4 cup sugar'},
    {'quantity': None,  'unit': None,   'item': 'water',         'original_text': 'water'},
]
converted = [convert_ingredient_to_si(i) for i in demo_ingredients]
pd.DataFrame(converted)[['item', 'quantity', 'unit', 'si_quantity', 'si_unit']]


In [ ]:
# Full SI conversion table
pd.DataFrame([
    {'unit': k, 'si_unit': v[0], 'factor': v[1]}
    for k, v in SI_CONVERSION.items()
])


## 4. Duplicate Detection

Recipes are fingerprinted with MD5 of `lower(name) + sorted(lower(ingredient items))`.
Exact text differences (punctuation, order of steps) don't affect the hash.

In [ ]:
from src.enrichment import content_hash

print('Content hashes per recipe:')
for r in recipes:
    h = content_hash(r)
    print(f"  {r['recipe_id']:8} {r['name']:<35} hash={h[:10]}...")


In [ ]:
# Run duplicate detection
recipes = detect_duplicates(recipes)

dups = [r for r in recipes if r['_meta']['is_duplicate']]
print(f'Found {len(dups)} duplicate(s):')
for d in dups:
    print(f"  {d['recipe_id']} — '{d['name']}' duplicates {d['_meta']['duplicate_of']}")


## 5. Auto-Tagging

In [ ]:
# Generate tags for each recipe
for r in recipes:
    r['tags'] = generate_tags(r)

tag_df = pd.DataFrame([
    {'recipe_id': r['recipe_id'], 'name': r['name'], 'tags': ', '.join(r['tags'])}
    for r in recipes
])
tag_df


In [ ]:
# Tag frequency distribution
from collections import Counter
all_tags = [tag for r in recipes for tag in r.get('tags', [])]
tag_counts = Counter(all_tags)
print('Tag distribution:')
for tag, count in tag_counts.most_common():
    bar = '█' * count
    print(f'  {tag:<20} {bar} ({count})')


## 6. Apply SI Conversion to All Recipes

In [ ]:
recipes = [convert_recipe_to_si(r) for r in recipes]

# Show ingredients of the first recipe after conversion
sample = recipes[0]
print(f"Sample: {sample['name']}")
df = pd.DataFrame(sample['ingredients'])
cols = [c for c in ['item','quantity','unit','si_quantity','si_unit','normalized_item'] if c in df.columns]
df[cols]


## 7. Save Enriched Data

In [ ]:
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)

print(f'✅ Saved {len(recipes)} enriched recipes → {OUTPUT_FILE}')
print('   Next step: run 03_validation.ipynb')
